# 04f — LoRA Training (L2 — Geometry (non-triangle-inequality))

**Runs on RunPod GPU.**

Trains Gemma-2-2B with PEFT LoRA on `lora_train_l2.jsonl` and saves the adapter.
After training, merges the adapter into base weights **on CPU** and saves the merged
checkpoint to `my_work/checkpoints/lora_l2_merged/`.

**Next step:** run `04g_lora_l2_attribution.ipynb` from a **fresh kernel** to load the
merged model and run attribution. Do NOT continue in this kernel — training leaves
~12 GB of CUDA memory that cannot be freed without a kernel restart.

**Prerequisites:** `04_lora_training_data.ipynb` must have been run first.

## 0 — Environment setup

In [1]:
import os
import sys
from pathlib import Path

# Must be set BEFORE torch is imported anywhere.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR.")

try:
    from dotenv import load_dotenv
    if _root is not None and (_root / ".env").is_file():
        load_dotenv(_root / ".env")
        print("Loaded .env")
except ImportError:
    pass

MY_WORK = _my_work if _root else Path(".").resolve()

if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    for key, path in {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
    }.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)
    print(f"RunPod HF cache: {hf_home}")

CHECKPOINT_DIR = MY_WORK / "checkpoints" / "lora_l2"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

STATS_DIR = MY_WORK / "results" / "statistics"
STATS_DIR.mkdir(parents=True, exist_ok=True)

GRAPHS_DIR = MY_WORK / "results" / "graphs_lora_l2"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"MY_WORK      : {MY_WORK}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")
print(f"Stats dir    : {STATS_DIR}")

Repo root: /workspace/thesis_circuit_breaker
RunPod HF cache: /workspace/hf
MY_WORK      : /workspace/thesis_circuit_breaker/my_work
Checkpoint   : /workspace/thesis_circuit_breaker/my_work/checkpoints/lora_l2
Stats dir    : /workspace/thesis_circuit_breaker/my_work/results/statistics


## 1 — Constants

In [2]:
import json
import time
import gc
import traceback
import random

import torch

# ── Model & tokenizer constants ────────────────────────────────────────────────
MODEL_NAME      = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"  # base CLT — unchanged from baseline
TOKEN_TRUE      = " True"
TOKEN_FALSE     = " False"
VOCAB_ID_TRUE   = 5569
VOCAB_ID_FALSE  = 7662

# ── LoRA hyperparameters ───────────────────────────────────────────────────────
LORA_R          = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "gate_proj", "up_proj"]

# ── Training hyperparameters ───────────────────────────────────────────────────
TRAIN_LR        = 2e-4
TRAIN_BATCH     = 4       # per-device; increase to 8 if VRAM allows
TRAIN_MAX_STEPS = 400     # ~400 steps on ~600 rows ≈ 2-3 epochs
TRAIN_WARMUP    = max(40, 400 // 10)      # 10% warmup
MAX_SEQ_LEN     = 128     # prompts are short; cap to save memory
SEED            = 42

PHASE = "lora_l2"

# ── Paths ──────────────────────────────────────────────────────────────────────
TRAIN_JSONL  = MY_WORK / "data" / "lora_train_l2.jsonl"
PROBE_JSONL  = MY_WORK / "data" / "prompts_triangle_v2.jsonl"

print(f"Model        : {MODEL_NAME}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}, target={LORA_TARGET_MODULES}")
print(f"Training     : lr={TRAIN_LR}, steps={TRAIN_MAX_STEPS}, batch={TRAIN_BATCH}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

Model        : google/gemma-2-2b
LoRA r=8, alpha=16, target=['q_proj', 'v_proj', 'gate_proj', 'up_proj']
Training     : lr=0.0002, steps=400, batch=4
CUDA         : True
GPU          : NVIDIA RTX A4500


## 2 — Load training data

In [3]:
train_rows = []
with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line.strip())
        if row.get("completion") in (" True", " False"):
            train_rows.append(row)

n_true  = sum(1 for r in train_rows if r["label"])
n_false = sum(1 for r in train_rows if not r["label"])
print(f"Training rows loaded: {len(train_rows)}")
print(f"  True : {n_true}  |  False: {n_false}")
print()
print(f"Probe JSONL  : {PROBE_JSONL}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")

Training rows loaded: 210
  True : 105  |  False: 105

Probe JSONL  : /workspace/thesis_circuit_breaker/my_work/data/prompts_triangle_v2.jsonl
Checkpoint   : /workspace/thesis_circuit_breaker/my_work/checkpoints/lora_l2


## 3 — Load HuggingFace model for LoRA training

We load the raw HuggingFace `AutoModelForCausalLM` (not the circuit-tracer
`ReplacementModel`) for training.  After training, the adapter is saved.
For attribution, we reload the circuit-tracer `ReplacementModel` and merge
the LoRA weights into it.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

device_str = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device_str)

print(f"Loading tokenizer from {MODEL_NAME} ...")
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if hf_tokenizer.pad_token is None:
    hf_tokenizer.pad_token = hf_tokenizer.eos_token
hf_tokenizer.padding_side = "left"  # Gemma uses left-padding for generation

print(f"Loading base model from {MODEL_NAME} ...")
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("Wrapping with LoRA ...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
lora_model = get_peft_model(hf_model, lora_config)
lora_model.print_trainable_parameters()

/workspace/venvs/ct/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading tokenizer from google/gemma-2-2b ...
Loading base model from google/gemma-2-2b ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Wrapping with LoRA ...
trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438


## 4 — Build tokenised dataset for training

In [5]:
from torch.utils.data import Dataset, DataLoader


class CompletionDataset(Dataset):
    """Tokenises (prompt, completion) pairs.  Loss is masked to completion only."""

    def __init__(self, rows, tokenizer, max_len):
        self.samples = []
        for row in rows:
            full_text = row["prompt"] + row["completion"]
            enc_full   = tokenizer(full_text,   truncation=True, max_length=max_len)
            enc_prompt = tokenizer(row["prompt"], truncation=True, max_length=max_len)
            n_prompt = len(enc_prompt["input_ids"])
            input_ids = enc_full["input_ids"]
            labels    = [-100] * n_prompt + input_ids[n_prompt:]
            if len(labels) < len(input_ids):
                labels = [-100] * len(input_ids)
            labels = labels[:len(input_ids)]
            self.samples.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels":    torch.tensor(labels,    dtype=torch.long),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    """Left-pad input_ids and labels to the longest sequence in the batch."""
    max_len = max(len(b["input_ids"]) for b in batch)
    padded_ids    = []
    padded_labels = []
    attn_masks    = []
    pad_id = hf_tokenizer.pad_token_id
    for b in batch:
        ids = b["input_ids"]
        lbl = b["labels"]
        pad_len = max_len - len(ids)
        padded_ids.append(torch.cat([torch.full((pad_len,), pad_id, dtype=torch.long), ids]))
        padded_labels.append(torch.cat([torch.full((pad_len,), -100, dtype=torch.long), lbl]))
        attn_masks.append(torch.cat([torch.zeros(pad_len, dtype=torch.long), torch.ones(len(ids), dtype=torch.long)]))
    return {
        "input_ids":      torch.stack(padded_ids),
        "labels":         torch.stack(padded_labels),
        "attention_mask": torch.stack(attn_masks),
    }


random.seed(SEED)
shuffled_rows = list(train_rows)
random.shuffle(shuffled_rows)

train_dataset = CompletionDataset(shuffled_rows, hf_tokenizer, MAX_SEQ_LEN)
train_loader  = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH,
    shuffle=True,
    collate_fn=collate_fn,
)

print(f"Dataset samples : {len(train_dataset)}")
print(f"Batches per step: {len(train_loader)} (batch_size={TRAIN_BATCH})")  
print(f"Target steps    : {TRAIN_MAX_STEPS}")

# Sanity: check completion token IDs
id_true  = hf_tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = hf_tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"Token mismatch: True  {id_true} != {VOCAB_ID_TRUE}"
assert id_false == VOCAB_ID_FALSE, f"Token mismatch: False {id_false} != {VOCAB_ID_FALSE}"
print(f"Token IDs verified: True={id_true}, False={id_false}")

Dataset samples : 210
Batches per step: 53 (batch_size=4)
Target steps    : 400
Token IDs verified: True=5569, False=7662


## 5 — Training loop

In [6]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR

optimizer = AdamW(lora_model.parameters(), lr=TRAIN_LR, weight_decay=0.01)

# Linear warmup then linear decay
warmup_scheduler = LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=TRAIN_WARMUP
)
decay_scheduler = LinearLR(
    optimizer, start_factor=1.0, end_factor=0.1, total_iters=TRAIN_MAX_STEPS - TRAIN_WARMUP
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, decay_scheduler],
    milestones=[TRAIN_WARMUP],
)

lora_model.train()
global_step = 0
running_loss = 0.0
log_every = 20
t0_train = time.time()

print(f"Starting training: {TRAIN_MAX_STEPS} steps, lr={TRAIN_LR}")
print()

loader_iter = iter(train_loader)

while global_step < TRAIN_MAX_STEPS:
    try:
        batch = next(loader_iter)
    except StopIteration:
        loader_iter = iter(train_loader)  # restart epoch
        batch = next(loader_iter)

    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()

    outputs = lora_model(**batch)
    loss = outputs.loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(lora_model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    running_loss += loss.item()
    global_step  += 1

    if global_step % log_every == 0 or global_step == TRAIN_MAX_STEPS:
        avg_loss = running_loss / log_every
        lr_now   = scheduler.get_last_lr()[0]
        elapsed  = time.time() - t0_train
        print(
            f"step {global_step:4d}/{TRAIN_MAX_STEPS}  "
            f"loss={avg_loss:.4f}  lr={lr_now:.2e}  "
            f"elapsed={elapsed:.0f}s"
        )
        running_loss = 0.0

print()
print(f"Training complete in {time.time() - t0_train:.0f}s.")

Starting training: 400 steps, lr=0.0002

step   20/400  loss=1.5149  lr=1.10e-04  elapsed=5s
step   40/400  loss=0.4995  lr=2.00e-04  elapsed=9s
step   60/400  loss=0.4784  lr=1.90e-04  elapsed=12s
step   80/400  loss=0.2953  lr=1.80e-04  elapsed=15s
step  100/400  loss=0.6302  lr=1.70e-04  elapsed=19s
step  120/400  loss=0.8566  lr=1.60e-04  elapsed=22s
step  140/400  loss=0.1450  lr=1.50e-04  elapsed=25s
step  160/400  loss=0.6467  lr=1.40e-04  elapsed=29s
step  180/400  loss=0.2616  lr=1.30e-04  elapsed=32s
step  200/400  loss=0.1966  lr=1.20e-04  elapsed=35s
step  220/400  loss=0.1154  lr=1.10e-04  elapsed=39s
step  240/400  loss=0.2437  lr=1.00e-04  elapsed=42s
step  260/400  loss=0.2796  lr=9.00e-05  elapsed=45s
step  280/400  loss=0.1584  lr=8.00e-05  elapsed=48s
step  300/400  loss=0.1526  lr=7.00e-05  elapsed=52s
step  320/400  loss=0.0464  lr=6.00e-05  elapsed=55s
step  340/400  loss=0.2086  lr=5.00e-05  elapsed=58s
step  360/400  loss=0.0192  lr=4.00e-05  elapsed=61s
step  3

## 6 — Save LoRA adapter

In [7]:
lora_model.save_pretrained(str(CHECKPOINT_DIR))
hf_tokenizer.save_pretrained(str(CHECKPOINT_DIR))

print(f"LoRA adapter saved to: {CHECKPOINT_DIR}")
print(f"  Files: {[f.name for f in CHECKPOINT_DIR.iterdir()]}")

LoRA adapter saved to: /workspace/thesis_circuit_breaker/my_work/checkpoints/lora_l2
  Files: ['tokenizer.json', 'tokenizer.model', 'special_tokens_map.json', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'README.md']


## 7 — Merge LoRA adapter into base weights (CPU only)

Loads base Gemma-2-2B on CPU, merges the adapter from `lora_l2/`,
and saves full merged weights to `lora_l2_merged/`.

**No GPU required for this step.**  After this cell, the pod can be stopped or
the kernel restarted.  `04g_lora_l2_attribution.ipynb` picks up from `lora_l2_merged/`.

In [9]:
from peft import PeftModel

print("Merging LoRA adapter into base model on CPU ...")
MERGED_DIR = MY_WORK / "checkpoints" / "lora_l2_merged"
MERGED_DIR.mkdir(parents=True, exist_ok=True)

_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cpu"
)
_peft = PeftModel.from_pretrained(_base, str(CHECKPOINT_DIR))
_merged = _peft.merge_and_unload()
_merged.save_pretrained(str(MERGED_DIR))
hf_tokenizer.save_pretrained(str(MERGED_DIR))
del _base, _peft, _merged
import gc; gc.collect()

print(f"Merged checkpoint saved → {MERGED_DIR}")
print("Done. Now open 04g_lora_l2_attribution.ipynb in a FRESH kernel.")


Merging LoRA adapter into base model on CPU ...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Merged checkpoint saved → /workspace/thesis_circuit_breaker/my_work/checkpoints/lora_l2_merged
Done. Now open 04g_lora_l2_attribution.ipynb in a FRESH kernel.
